# 🏆 Flagship Lab: Fun & Fit Health Advisor Agent (GitHub Models)

## 🧑‍🏫 Scenario
You are building an AI Health Advisor for a busy professional who:
- Has limited time
- Wants to stay fit
- Needs quick, actionable advice

Your goal is to build a **real AI Agent** step-by-step.

---

## 🎯 What You Will Learn
- What makes an Agent different from an LLM
- How Agents decide actions
- How tool calling works internally
- How to build a multi-step reasoning loop
- How to extend to real-world apps

---

## 🔧 Step 1: Setup GitHub Models

👉 We replace Azure with GitHub Models (same capability, simpler setup)


In [ ]:
from openai import OpenAI
import os

client = OpenAI(
    api_key=os.getenv("GITHUB_TOKEN"),
    base_url="https://models.inference.ai.azure.com"
)

model = "gpt-4.1"
print("✅ Connected to GitHub Models")

## 💬 Step 2: Baseline (Normal LLM)

Let's see how a normal model behaves.

👉 Observe: No reasoning, no tools


In [ ]:
response = client.chat.completions.create(
    model=model,
    messages=[{"role":"user","content":"I want to stay fit"}]
)
print(response.choices[0].message.content)

🔍 **Observation Task:**
- Is it personalized?
- Does it calculate anything?

---

## 🧠 Step 3: Add Agent Personality

We now control behavior using system prompt.


In [ ]:
system_prompt = """
You are a Fun & Fit Health Advisor Agent.

Rules:
- Friendly & motivating
- Short actionable advice
- If calculation needed → use tools
- Always personalize response
"""

In [ ]:
response = client.chat.completions.create(
    model=model,
    messages=[
        {"role":"system","content":system_prompt},
        {"role":"user","content":"I want to lose weight"}
    ]
)
print(response.choices[0].message.content)

🔍 **Observation:** Behavior changed → This is controllable AI

---

## 🛠️ Step 4: Tools (Superpower of Agents)

Agents can:
- Compute
- Call APIs
- Fetch data

👉 We start with BMI tool


In [ ]:
def calculate_bmi(weight, height):
    bmi = weight/(height**2)

    if bmi < 18.5:
        category="Underweight"
    elif bmi < 25:
        category="Normal"
    elif bmi < 30:
        category="Overweight"
    else:
        category="Obese"

    return f"BMI: {round(bmi,2)} ({category})"


In [ ]:
tools=[{
 "type":"function",
 "function":{
  "name":"calculate_bmi",
  "description":"Calculate BMI",
  "parameters":{
    "type":"object",
    "properties":{
      "weight":{"type":"number"},
      "height":{"type":"number"}
    },
    "required":["weight","height"]
  }
 }
}]

## ⚙️ Step 5: Let Model Decide Tool Usage

👉 Model decides whether to call function


In [ ]:
response = client.chat.completions.create(
 model=model,
 messages=[
  {"role":"system","content":system_prompt},
  {"role":"user","content":"My weight is 90kg and height is 1.8m"}
 ],
 tools=tools
)

print(response)

🔍 **Observation:**
- Did model call tool?
- What arguments?

---

## 🔁 Step 6: Core Agent Loop (MOST IMPORTANT)

This is what Azure Agents do internally.

We now build it manually.


In [ ]:
import json

messages=[
 {"role":"system","content":system_prompt},
 {"role":"user","content":"My weight is 90kg and height is 1.8m. Give advice"}
]

while True:
    response=client.chat.completions.create(
        model=model,
        messages=messages,
        tools=tools
    )

    msg=response.choices[0].message

    if not msg.tool_calls:
        print("\n✅ Final Answer:\n")
        print(msg.content)
        break

    tool_call=msg.tool_calls[0]
    args=json.loads(tool_call.function.arguments)

    print("\n🔧 Tool Call Detected")
    print("Function:", tool_call.function.name)
    print("Input:", args)

    result=calculate_bmi(**args)

    print("Output:", result)

    messages.append(msg)
    messages.append({
        "role":"tool",
        "tool_call_id":tool_call.id,
        "content":result
    })

## 🚀 Step 7: Extend Agent (Challenge)

Add new tools:
- Calorie calculator
- Workout planner

👉 Make agent smarter


## ⚖️ Step 8: Agent vs LLM

| Feature | LLM | Agent |
|--------|-----|--------|
| Static answers | ✅ | ✅ |
| Tool usage | ❌ | ✅ |
| Multi-step reasoning | ❌ | ✅ |

---

## 🏁 Final Challenge

Build a complete assistant:
- Takes user profile
- Calculates BMI
- Suggests diet
- Suggests workout

🎯 This is a real-world AI Agent!
